# General structure discharge comparison

Compare the total discharge through the `Maeslantkering` general structure for the current `input_work` results and the Windows reference results.

**Setup**: Just specify the test number (e.g., "c028") in the `TEST_NUMBER` variable, and the notebook will automatically locate the test case directory and find the `_his.nc` history files in the output directories.

In [ ]:
from pathlib import Path

%matplotlib tk
import matplotlib.pyplot as plt
from netCDF4 import Dataset, chartostring, num2date

# Configuration: specify only the test number (e.g., "c028")
TEST_NUMBER = "c031"
TEST_LOCATION = "d:/models/unst/08868"

# Build CASE_DIR from test number
CASE_DIR = Path(f"{TEST_LOCATION}/{TEST_NUMBER}_*").expanduser()
matching_dirs = list(Path(TEST_LOCATION).glob(f"{TEST_NUMBER}_*"))
if matching_dirs:
    CASE_DIR = matching_dirs[0]
else:
    raise FileNotFoundError(f"No test case found matching pattern: {TEST_NUMBER}_*")

# Function to find the _his.nc file in an output directory
def find_history_file(output_dir):
    """Find the _his.nc file in the given output directory."""
    his_files = list(output_dir.glob("*_his.nc"))
    if not his_files:
        raise FileNotFoundError(f"No _his.nc file found in {output_dir}")
    return his_files[0]

# Build RESULT_FILES by finding _his.nc files in each variant
RESULT_FILES = {}
# for variant in ["input_work", "reference_win64", "input_v0"]:
for variant in ["input_work", "input_v0"]:
    output_dir = CASE_DIR / variant / "dflowfm" / "dflowfmoutput"
    if output_dir.exists():
        history_file = find_history_file(output_dir)
        RESULT_FILES[variant] = history_file

missing_files = [path for path in RESULT_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Missing result files:\n" + "\n".join(map(str, missing_files)))

In [86]:
import pandas as pd


GENERAL_STRUCTURE_PLOT_GROUPS = (
    {
        "title": "Discharge components",
        "ylabel": "Discharge (m3/s)",
        "variables": {
            "general_structure_discharge": "total",
            # "general_structure_discharge_through_gate_opening": "through gate opening",
            # "general_structure_discharge_over_gate": "over gate",
            # "general_structure_discharge_under_gate": "under gate",
        },
    },
    {
        "title": "Water levels",
        "ylabel": "Level (m)",
        "variables": {
            "general_structure_s1up": "upstream level",
            "general_structure_s1dn": "downstream level",
            # "general_structure_head": "head difference",
            # "general_structure_s1_on_crest": "level on crest",
        },
    },
    {
        "title": "Structure elevations",
        "ylabel": "Elevation (m)",
        "variables": {
            "general_structure_crest_level": "crest level",
            "general_structure_gate_lower_edge_level": "gate lower edge",
            "general_structure_gate_upper_edge_level": "gate upper edge",
        },
    },
    {
        "title": "Structure dimensions",
        "ylabel": "Length (m)",
        "variables": {
            "general_structure_crest_width": "crest width",
            # "general_structure_gate_height": "gate height",
            # "general_structure_gate_opening_height": "gate opening height",
            "general_structure_gate_opening_width": "gate opening width",
        },
    },
    {
        "title": "Flow areas",
        "ylabel": "Area (m2)",
        "variables": {
            "general_structure_flow_area": "total",
            # "general_structure_flow_area_in_gate_opening": "in gate opening",
            # "general_structure_flow_area_over_gate": "over gate",
            # "general_structure_flow_area_under_gate": "under gate",
        },
    },
    {
        "title": "Velocities",
        "ylabel": "Velocity (m/s)",
        "variables": {
            "general_structure_velocity": "total",
            # "general_structure_velocity_through_gate_opening": "through gate opening",
            # "general_structure_velocity_over_gate": "over gate",
            # "general_structure_velocity_under_gate": "under gate",
        },
    },
    # {
    #     "title": "Flow state",
    #     "ylabel": "State (-)",
    #     "variables": {
    #         "general_structure_state": "state",
    #     },
    # },
    # {
    #     "title": "Force difference",
    #     "ylabel": "Force per unit length (N/m)",
    #     "variables": {
    #         "general_structure_force_difference": "force difference",
    #     },
    # },
)


def decode_structure_names(variable):
    """Decode a NetCDF structure-name or structure-ID variable into labels."""
    values = variable[:]
    if values.dtype.kind in {"S", "U"}:
        values = chartostring(values).tolist()
    else:
        values = values.tolist()
    return [str(value).strip() for value in values]


def read_general_structure_data(history_file):
    """Return all available general-structure time series from a history file."""
    variable_names = {
        variable_name
        for group in GENERAL_STRUCTURE_PLOT_GROUPS
        for variable_name in group["variables"]
    }
    with Dataset(history_file) as dataset:
        variables = dataset.variables
        name_variable = variables.get("general_structure_name")
        if name_variable is None:
            name_variable = variables["general_structure_id"]
        structure_names = decode_structure_names(name_variable)
        time = variables["time"]
        timestamps = num2date(
            time[:],
            units=time.units,
            calendar=getattr(time, "calendar", "standard"),
        )
        output_values = {
            variable_name: variables[variable_name][:]
            for variable_name in variable_names
            if variable_name in variables
        }

    timestamps_dt = pd.to_datetime([str(timestamp) for timestamp in timestamps])
    return {
        structure_name: {
            "timestamps": timestamps_dt,
            "variables": {
                variable_name: values[:, structure_index]
                for variable_name, values in output_values.items()
            },
        }
        for structure_index, structure_name in enumerate(structure_names)
    }


structure_data = {
    label: read_general_structure_data(history_file)
    for label, history_file in RESULT_FILES.items()
}
structure_names = sorted({
    structure_name
    for data in structure_data.values()
    for structure_name in data
})

In [87]:
variant_line_styles = ("-", ":", "--", "-.")
line_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
plot_rows = 2
plot_columns = 3
plots_per_figure = plot_rows * plot_columns

for structure_name in structure_names:
    available_groups = [
        group
        for group in GENERAL_STRUCTURE_PLOT_GROUPS
        if any(
            variable_name in structure["variables"]
            for data in structure_data.values()
            if (structure := data.get(structure_name)) is not None
            for variable_name in group["variables"]
        )
    ]
    if not available_groups:
        continue

    page_count = (len(available_groups) + plots_per_figure - 1) // plots_per_figure
    for page_index in range(page_count):
        page_start = page_index * plots_per_figure
        page_groups = available_groups[page_start:page_start + plots_per_figure]
        figure_name = (
            structure_name
            if page_count == 1
            else f"{structure_name} ({page_index + 1}/{page_count})"
        )
        fig, axes = plt.subplots(
            plot_rows,
            plot_columns,
            figsize=(18, 9),
            sharex=True,
            squeeze=False,
            num=figure_name,
            clear=True,
        )

        for axis, group in zip(axes.flat, page_groups):
            for variant_index, (label, data) in enumerate(structure_data.items()):
                structure = data.get(structure_name)
                if structure is None:
                    continue

                for variable_index, (variable_name, series_label) in enumerate(group["variables"].items()):
                    values = structure["variables"].get(variable_name)
                    if values is None:
                        continue
                    color_index = variable_index + variant_index * len(group["variables"])
                    axis.plot(
                        structure["timestamps"],
                        values,
                        label=f"{label}: {series_label}",
                        color=line_colors[color_index % len(line_colors)],
                        linewidth=2.5 if variant_index == 0 else 1.75,
                        linestyle=variant_line_styles[variant_index % len(variant_line_styles)],
                        zorder=variant_index + 1,
                        drawstyle="steps-post" if variable_name == "general_structure_state" else "default",
                    )

            axis.set(
                title=group["title"],
                ylabel=group["ylabel"],
            )
            axis.grid(True, alpha=0.3)
            axis.legend(fontsize="small")

        for axis in axes.flat[len(page_groups):]:
            axis.set_visible(False)

        page_suffix = f" ({page_index + 1}/{page_count})" if page_count > 1 else ""
        fig.suptitle(f"General structure: {structure_name}{page_suffix}")
        fig.supxlabel("Time")
        fig.autofmt_xdate()
        fig.tight_layout()
        plt.show()